# 03 · The service: API, privacy and an optional LLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammedelsayed741996-blip/berlin-data-portfolio/blob/main/hr-tech/hr-policy-rag-assistant/notebooks/03_service_privacy_and_llm.ipynb)

The assistant runs as a small web service (Starlette and uvicorn) with a TypeScript front end, in
Docker, with PostgreSQL for the question log and Kubernetes manifests for deployment. This
notebook calls the API in-process, so it runs anywhere, including Colab.

In [1]:
import os, sys, subprocess, warnings
from pathlib import Path

REPO = "https://github.com/mohammedelsayed741996-blip/berlin-data-portfolio.git"
PROJECT = "hr-tech/hr-policy-rag-assistant"

if "google.colab" in sys.modules:
    if not Path("/content/berlin-data-portfolio").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/berlin-data-portfolio"], check=True)
    os.chdir(f"/content/berlin-data-portfolio/{PROJECT}")
else:
    here = Path.cwd()
    while not (here / "src" / "hrrag").exists() and here != here.parent:
        here = here.parent
    os.chdir(here)

sys.path.insert(0, str(Path.cwd() / "src"))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)
from IPython.display import Image, display
print("Project folder:", Path.cwd().name)

Project folder: hr-policy-rag-assistant


In [2]:
import asyncio, json
from hrrag.api import create_app
from hrrag.config import Settings

app = create_app(Settings(database_url="sqlite:///:memory:"))

def call(method, path, body=None, headers=None, query=""):
    """Call the ASGI app directly, the way a test client would."""
    data = json.dumps(body).encode() if body is not None else b""
    msgs, sent = [], False
    async def receive():
        nonlocal sent
        if not sent:
            sent = True
            return {"type": "http.request", "body": data, "more_body": False}
        return {"type": "http.disconnect"}
    async def send(m):
        msgs.append(m)
    hdrs = [(b"content-type", b"application/json")] + [(k.lower().encode(), v.encode()) for k, v in (headers or {}).items()]
    scope = {"type": "http", "method": method, "path": path, "query_string": query.encode(), "headers": hdrs,
             "http_version": "1.1", "scheme": "http", "server": ("nb", 80), "client": ("127.0.0.1", 1), "root_path": ""}
    asyncio.run(app(scope, receive, send))
    status = next(m["status"] for m in msgs if m["type"] == "http.response.start")
    return status, json.loads(b"".join(m.get("body", b"") for m in msgs if m["type"] == "http.response.body"))

call("GET", "/health")

(200, {'status': 'ok', 'passages': 52, 'engine': 'extractive', 'demo_mode': True})

## Personal data never reaches the log

Employees sometimes paste their own details into a question. Names, employee IDs, emails, phone
numbers, IBANs, tax IDs and dates are replaced before the question is answered or stored, and the
log is deleted after 90 days.

In [3]:
status, out = call("POST", "/api/ask", {"question": "Hi, my name is Lena Fischer (E-204871), how long can I take Elternzeit?", "session": "nb"})
print(out["answer"]); print("removed:", out["redacted"])
print("stored:", app.state.store._exec("SELECT question FROM queries").fetchall())

Each parent may take up to three years of parental leave per child. [1]
removed: ['EMPLOYEE_ID', 'NAME']
stored: [('Hi, my name is [NAME] ([EMPLOYEE_ID]), how long can I take Elternzeit?',)]


## The role comes from the identity proxy, not the browser

In production a single sign-on proxy sets the `X-User-Role` header. A request that claims to be
HR in its body is still treated as an employee when the header says so.

In [4]:
for headers in ({"X-User-Role": "employee"}, {"X-User-Role": "hr"}):
    _, out = call("POST", "/api/ask", {"question": "What is the merit budget for 2026?", "role": "hr"}, headers=headers)
    print(headers["X-User-Role"], "->", out["answered"], "|", out["answer"][:90])

employee -> False | I could not find this in the HR policies I have access to. Please ask your HR business par
hr -> True | The merit increase budget for 2026 is 3.5 percent of the base salary sum of each departmen


## Plugging in an LLM, with guardrails

Set `LLM_BASE_URL`, `LLM_API_KEY` and `LLM_MODEL` to use any OpenAI-compatible model. The prompt
allows only the numbered passages, and the reply is checked: an answer without a citation, or with a
citation to a passage it was not given, is replaced by the extractive answer. Here a fake model
shows each case.

In [5]:
from hrrag.answer import Assistant, LLMAnswerer
from hrrag.corpus import load
from hrrag.retrieval import Retriever

retriever = Retriever(load(Path("policies")))
def fake(reply):
    return LLMAnswerer(Settings(llm_base_url="http://llm", llm_model="demo"), transport=lambda payload: {"choices": [{"message": {"content": reply}}]})

for reply in ["Office staff can work remotely up to two days a week [1].", "Probably three days or so.", "Two days [7].", "NOT_FOUND"]:
    a = Assistant(retriever, answerer=fake(reply)).ask("How many days a week can I work remotely?")
    print(f"model said: {reply!r:60} -> answered={a.answered}: {a.text[:70]}")

model said: 'Office staff can work remotely up to two days a week [1].'  -> answered=True: Office staff can work remotely up to two days a week [1].
model said: 'Probably three days or so.'                                 -> answered=True: Employees in office roles may work remotely for up to two days per wee
model said: 'Two days [7].'                                              -> answered=True: Employees in office roles may work remotely for up to two days per wee
model said: 'NOT_FOUND'                                                  -> answered=False: I could not find this in the HR policies I have access to. Please ask 


In [6]:
print(LLMAnswerer.prompt("How many days a week can I work remotely?", retriever.search("remote work days", k=2)[0])[0]["content"])

You answer employees' questions about company HR policies. Use only the numbered passages. Cite every statement with its passage number in square brackets, like [1]. Keep the answer under 80 words. If the passages do not contain the answer, reply exactly: NOT_FOUND. Never guess numbers or dates.


## Running it for real

```bash
docker compose up --build          # the app and PostgreSQL, on http://localhost:8000
kubectl apply -f deploy/k8s/        # Deployment, Service, ConfigMap, autoscaler (add the Secret first)
```

CI builds the image, starts it, and checks `/health` and a real question against the running
container. It also runs the test suite against a PostgreSQL service.